[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gabraxas/LVs-and-Policy/blob/main/lecture/week06/week06_structures.ipynb)

# 6주차 · 발사체의 하중과 구조, 그리고 탱크 만들기
### Loads, Structures & Propellant Tank Manufacturing — 음료캔보다 얇은 껍데기가 수백 톤을 버티는 이유

**경상국립대학교 · 우주수송정책및발사체기술**

> 슈퍼마켓 음료캔을 손가락으로 옆에서 누르면 쉽게 찌그러집니다. 그런데 뚜껑을 따지 않은 캔 위에는 어른이 올라서도 버팁니다. 안쪽의 압력이 얇은 벽을 팽팽하게 잡아 주기 때문입니다. 오늘 이야기의 주인공인 발사체 추진제 탱크는 이 음료캔보다 **상대적으로 더 얇은 껍데기**이고, 그 껍데기가 수백 톤의 추진제와 상단을 떠받친 채 음속을 돌파합니다. 오늘은 그 껍데기가 **무엇에 시달리는지(하중)**, **어떻게 버티게 만드는지(구조와 재료)**, 그리고 **공장에서 어떻게 태어나 시험으로 증명되는지(제작과 개발)**를 하나의 이야기로 따라가 봅니다.

- 📎 4주차 보충자료 `week04/week04-loads-analysis.ipynb`(하중해석과 예비 구조설계)를 바탕으로, 쉬운 설명·실습·제작 이야기를 더해 재구성한 강의자료임
- 예제 발사체는 4주차에서 외형까지 계산한 **LV-B** (지름 3.7 m, 전장 43.1 m, GLOW 249 t, 케로신/LOX 2단)를 그대로 사용함
- 수식 유도와 심화 내용은 `▶ 자세히 보기` 접기 안에 넣었음 — 본문만 읽어도 흐름이 이어지도록 구성

In [ ]:
# ▶ 실행 전 준비 — 이 셀을 먼저 실행하세요 (약 10~20초 소요)
!pip install -q ipywidgets
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/_shared/course_interactive.py",
    "course_interactive.py")

from course_interactive import *
setup_korean_font()
print("준비 완료 — 아래 셀들을 순서대로 실행하며 강의를 진행하세요.")

## 학습 목표

이번 주가 끝나면 다음을 설명하고 계산할 수 있어야 함

1. 발사체가 **언제, 무엇 때문에** 가장 큰 하중을 받는지 비행 단계별로 설명 (지상풍 → 점화 → Max-Q → 연소종료 → 단분리)
2. 기체를 **막대(보) 하나**로 보고 축력·전단력·굽힘모멘트를 읽어 내고, 이를 **등가 압축하중**으로 바꾸는 방법
3. 설계하중이 **99/90 통계 기준**과 **안전계수**로 정해지는 이유와, 데이터 양이 마진에 미치는 영향
4. 얇은 탱크가 실패하는 세 가지 방식(**터짐 · 좌굴 · 국부 접힘**)과 **내압이 구조를 돕는 원리(풍선 효과)**
5. 재료(알루미늄 합금, Al-Li, 스테인리스, 복합재)와 구조 형식(모노코크, 스트링거, 이소그리드)의 선택 논리
6. 추진제 탱크의 **제작 공정**(깎기 → 굽히기 → 용접 → 검사 → 압력시험)과 **개발 시험**의 단계
7. 구조 설계 선택이 **사업성·제조 인프라·산업정책**으로 이어지는 연결고리

## 오늘의 이야기 흐름

| 장 | 질문 | 핵심 도구 |
|---|---|---|
| 프롤로그 | 발사체 탱크는 얼마나 얇은가? | D/t 비교 |
| PART I · 로켓의 하루 | 발사체는 언제, 무엇에 시달리는가? | 비행 하중 타임라인 실습 |
| PART II · 막대 하나로 보기 | 하중을 어떻게 숫자로 바꾸는가? | 단면 절단 실습, 등가하중 |
| PART III · 얼마나 튼튼하게 | '최악의 경우'는 어떻게 정하는가? | 99/90 몬테카를로 실습 |
| PART IV · 어떻게 버티게 | 어떤 형식과 재료로 버티는가? | 좌굴·재료·이소그리드 실습 |
| PART V · 공장 이야기 | 탱크는 어떻게 만들어지는가? | 제작 공정도, 용접 이야기 |
| PART VI · 시험 이야기 | 버틴다는 것을 어떻게 증명하는가? | 시험 피라미드, 실패 사례, 피로 실습 |
| PART VII · 정책과 사업의 눈 | 구조 선택은 왜 사업·정책 문제인가? | 토론 |

---
# 프롤로그 · 음료캔보다 얇은 로켓

> 1957년 처음 날아오른 미국의 아틀라스 미사일은 탱크 벽이 가장 얇은 곳에서 0.36 mm에 불과했습니다. 그래서 아틀라스는 공장에서 발사대까지 늘 질소로 부풀린 채 옮겨야 했고, 압력이 빠지면 제 무게를 이기지 못하고 주저앉았습니다. 실제로 1963년 반덴버그 기지에서 정적 시험 중이던 아틀라스-아제나 기체가 가압 계통 고장으로 통째로 무너진 일이 있었습니다. 극단적인 예처럼 보이지만, 이 '풍선 탱크'는 오늘 배울 거의 모든 개념 — 하중, 좌굴, 내압의 도움, 재료, 제작 — 을 한 몸에 담고 있습니다.

In [ ]:
w6_thinness_ratio()

- **D/t (지름 ÷ 두께)** 가 클수록 상대적으로 얇은 껍데기 → 발사체 탱크는 D/t 수백~수천 수준
- 이렇게 얇은데도 버티는 **세 가지 비결**이 오늘 강의 전체의 뼈대
  - ① **내압** — 탱크 안의 압력이 벽을 팽팽하게 당겨 찌그러짐을 막음 (PART II·IV)
  - ② **보강 형식** — 격자(이소그리드)나 보강재로 같은 무게에서 강성을 몇 배로 높임 (PART IV)
  - ③ **재료와 제작** — 가볍고 강한 합금을, 강도를 잃지 않게 이어 붙이는 기술 (PART IV·V)
- 왜 이렇게까지 얇게 만드는가 → 4주차에서 본 것처럼 **상단 구조질량 1 kg은 탑재체 1 kg과 거의 1:1로 경쟁**하기 때문

> **핵심 개념**: 발사체 구조 설계는 "충분히 튼튼하게"가 아니라 "꼭 필요한 만큼만 튼튼하게"를 찾는 일입니다. 그 '꼭 필요한 만큼'을 정하는 출발점이 하중입니다.

---
# PART I · 로켓의 하루 — 하중은 언제, 어디서 오는가

## 1.1 하중의 세 얼굴 — 정적 · 동적 · 열

- **정적·준정적 하중**: 천천히 변해서 '무게처럼' 작용하는 힘
  - 추력에 의한 축 가속(관성 하중), 지상풍이 미는 힘, 탱크 내압
- **동적 하중**: 빠르게 흔들리는 힘 — 구조의 고유진동과 맞물리면 증폭됨
  - 점화 과압, 음향·랜덤 진동, 돌풍, 버펫, 슬로싱, 추력 진동(POGO), 분리 충격
- **열 하중**: 온도 차이가 만드는 힘
  - 극저온 추진제(LOX −183 °C)로 인한 수축, 공력가열, 엔진 복사열
- 지배적인 하중은 **비행 단계마다 바뀜** → 구조의 각 부분은 서로 다른 '최악의 순간'을 가짐

<div align="center"><img src="https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week04/images/loads-fig-03.png" width="820"></div>

<div align='center'><i>비행 단계별 하중 유형과 물리적 특성</i></div>

## 1.2 발사 당일 시간표 — 로켓의 하루를 따라가 보기

| 시각 | 사건 | 무엇이 구조를 괴롭히나 | 주로 영향받는 곳 |
|---|---|---|---|
| 발사 수일 전 | 공장 → 발사장 수송, 기립 | 수송 진동, 들어 올릴 때의 국부 하중 | 인양 부착점, 무가압 탱크 |
| T−수 시간 | 발사대 대기 | **지상풍** — 기체가 깃대처럼 흔들림, 바람 뒤 소용돌이 진동 | 기저부 굽힘, 고정 장치 |
| T−0 | 엔진 점화 | **점화 과압(IOP)**, 화염덕트 반사파, 굉음 | 노즐·엔진부, 하부 외피 |
| T+약 1분 | 천음속 통과, **최대동압(Max-Q)** | 공기력 최대 + 고층 바람 → 굽힘 최대, 버펫 | 기체 전체, 특히 탱크·단간부 |
| 1단 연소종료 직전 | 추진제 거의 소진 | **축 가속 최대** (예제 4.8 g) | 상단·탑재체 부착부 |
| 단분리 | 분리 장치 작동 | **충격(shock)**, 순간 하중 변화 | 분리면, 전자장비 |

- 하중 담당자는 이 시간표 전체에서 **각 부위별 최악의 순간**을 골라 설계하중으로 삼음

<div align="center"><img src="https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week04/images/loads-fig-04.png" width="820"></div>

<div align='center'><i>비행 단계별 하중 이벤트 타임라인</i></div>

## 1.3 [인터랙티브] 비행 하중 타임라인 — LV-B의 1단 비행 150초

- 아래 실습은 LV-B 1단 상승을 단순 중력선회로 계산해 **세 가지 하중 지표**의 시간이력을 보여줌

$$
q = \tfrac{1}{2}\rho V^2, \qquad n_x = \frac{T - D}{m\,g_0}, \qquad q\alpha = q \times \alpha_{\text{바람}}
$$

- **동압 q**: 공기가 기체를 미는 세기 — 속도는 빨라지는데 공기는 희박해지므로 중간에 **최대점(Max-Q)** 이 생김
- **축 하중배수 nₓ**: 위에 얹힌 질량을 '몇 g로' 누르는가 — 추진제가 줄수록 커져 **연소종료 직전 최대**
- **qα**: 바람 때문에 생긴 받음각 α와 동압의 곱 — **옆으로 꺾는 굽힘**의 대표 지표, 고층 제트기류 고도에서 최대

**해 볼 것**
- 제트기류 풍속을 0 → 80 m/s로 올리면 qα의 봉우리가 어떻게 변하는가? (q와 nₓ는 그대로인 이유는?)
- 이륙 T/W를 1.6 이상으로 올리면 Max-Q는 어떻게 되는가? — 빨리 가속할수록 짙은 공기 속에서 빠르게 날게 됨

In [ ]:
w6_flight_loads_explorer()

- **읽는 법**: 세 그래프의 봉우리가 **서로 다른 시각**에 있음 → 한 가지 '최악의 순간'은 없음
  - Max-Q 근처: 굽힘·외피 압력이 지배 → 탱크와 단간부의 설계 조건
  - 연소종료 직전: 축 압축이 지배 → 상단 지지 구조, 탑재체 어댑터의 설계 조건
- 실제 발사체는 Max-Q 근처에서 **추력을 줄여(스로틀 버킷)** 동압을 낮추기도 함 (4주차 실습 참고)
- 발사 당일 **고층 바람 관측**으로 qα를 다시 계산해 한계를 넘으면 발사를 연기 → 구조 마진이 곧 **발사 가능일수**

<details>
<summary>▶ 자세히 보기 — 이 계산 모델의 가정 (선택)</summary>

- 2차원 평면 비행, 지구 곡률·중력 감소만 반영, 받음각 0의 중력선회 (속도 80 m/s에서 2.5° 기울임)
- 대기: 지수 대기 모델(밀도 척도고도 8.5 km), 음속은 11 km까지 선형 감소
- 항력계수: 아음속 0.25 → 천음속 0.5 봉우리 → 초음속 감소의 단순 곡선
- 바람: 11 km 고도에 중심을 둔 가우스 분포 제트기류, 받음각 = 바람의 수직 성분 / 비행속도
- 추력: 해면 추력(T/W × GLOW)에서 진공 추력(×1.10)까지 외기압에 비례해 증가
- 목적은 **봉우리의 위치와 크기 감각**이며, 실제 하중해석은 6자유도 궤적 + 측정 바람 프로파일 수천 개로 수행

</details>

## 1.4 대기 속의 세 골칫거리 — 정적 공력 · 돌풍 · 버펫, 그리고 슬로싱

| 현상 | 쉬운 비유 | 특징 | 대책 |
|---|---|---|---|
| **정적 공력(받음각) 하중** | 비스듬히 든 우산에 바람이 걸리는 것 | 느리게 변함, 굽힘의 '기본값' | 바람에 맞춘 비행 프로그램(바람 편향) |
| **돌풍(gust)** | 고속도로에서 옆바람에 차가 휘청 | 순간적, 10~20 Hz 대역 응답 | 통계적 하중 정의(PART III) |
| **버펫(buffet)** | 깃발이 펄럭이는 것 | 형상 급변부의 유동박리·충격파 진동 | 페어링 형상 개선, 풍동시험 |
| **슬로싱(slosh)** | 컵 속 물이 출렁이는 것 | 탱크 내 자유표면 진동 → 자세제어와 결합 | 배플(baffle), 제어기 노치 필터 |

- 버펫이 기체의 굽힘 고유진동과 **같은 주파수로 맞물리면** 진동이 커져 치명적 → 형상과 강성을 함께 설계
- 슬로싱은 구조 문제이자 **제어 문제** — 5주차 GNC의 자세제어 루프와 연결됨

<div align="center"><img src="https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week04/images/loads-fig-08.png" width="760"></div>

<div align='center'><i>천음속 구간의 세 가지 대기 하중 상호작용</i></div>

<div align="center"><img src="https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week04/images/loads-fig-09.png" width="760"></div>

<div align='center'><i>배플 유무에 따른 슬로싱 거동 비교</i></div>

---
# PART II · 막대 하나로 보기 — 하중을 숫자로 바꾸는 법

## 2.1 빵을 썰듯이 — 단면 절단(section cut)의 아이디어

> 긴 바게트를 아무 데서나 칼로 자른다고 상상해 보세요. 자른 단면의 위쪽 조각이 아래쪽 조각을 얼마나 세게 누르고, 옆으로 얼마나 밀고, 얼마나 비틀어 꺾으려 하는지를 알면 그 단면이 견뎌야 할 힘을 알 수 있습니다. 하중 엔지니어는 발사체를 기저부부터 기수까지 수십 군데(스테이션)에서 이렇게 가상으로 잘라 봅니다.

- 예비설계에서는 발사체를 **휘지 않는 막대(강체 보)** 로 단순화
- 각 스테이션에서 세 가지 **내력**을 계산
  - **축력 N**: 위쪽이 아래쪽을 길이 방향으로 누르는 힘
  - **전단력 V**: 위쪽이 아래쪽을 옆으로 미는 힘
  - **굽힘모멘트 M**: 위쪽이 아래쪽을 꺾으려는 힘 — 전단력을 기수부터 적분한 값

$$
N(x) = D + a_x\, m_{\text{위}}(x), \qquad a_x = \frac{T - D}{m_{\text{전체}}}
$$

- **놀라운 사실**: 축력 식에 **중력이 없음** — 비행 중 기체 전체가 함께 떨어지므로 중력은 내력을 만들지 않음 (자유낙하하는 엘리베이터 안에서 몸무게가 0이 되는 것과 같은 원리)
  - 위 질량을 누르는 것은 **추력이 만든 가속**뿐 → 그래서 연소종료 직전 가속이 가장 클 때 축 하중이 최대
- 옆 방향은 **기수부 공기력**과 이를 상쇄하는 **엔진 짐벌 횡력**이 짝을 이루고, 그 사이에서 굽힘모멘트가 가장 커짐

<div align="center"><img src="https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week04/images/loads-fig-13.png" width="820"></div>

<div align='center'><i>스테이션 A~D 단면의 축력·전단력·굽힘모멘트 분포</i></div>

## 2.2 [인터랙티브] LV-B를 썰어 보자 — 단면 내력 탐색기

- 4주차 외형 스택업(엔진부 → RP-1 → LOX → 단간부 → 2단 → 페어링)에 질량을 배분한 모델
- 슬라이더: **동압 q, 받음각 α, 1단 추진제 잔량, 탱크 내압**
- 오른쪽 끝 그림 ④는 축력과 굽힘을 하나로 합친 **등가 압축하중**과, 내압이 그것을 얼마나 덜어 주는지를 보여줌

**해 볼 것**
- α = 0으로 두면 전단력·굽힘모멘트가 모두 사라지는가? 축력은?
- 추진제 잔량을 1.0 → 0.1로 줄이면 1단 하부의 축력이 줄어드는가 늘어나는가? (총질량 ↓ vs 가속 ↑)
- 탱크 내압을 0으로 내리면 탱크 구간의 등가하중은 몇 배가 되는가?

In [ ]:
w6_section_loads_explorer()

## 2.3 등가하중 — 축력과 굽힘을 '누르는 힘 하나'로

- 굽힘모멘트는 원통 단면의 한쪽을 **누르고**, 반대쪽을 **당김** → 눌리는 쪽이 축력과 겹쳐 가장 위험
- 그래서 예비설계에서는 둘을 합쳐 **가장 불리한 방위각의 압축하중 하나**로 바꿔서 봄

$$
P_{eq} = N + \frac{2M}{r} \qquad (\text{얇은 원통 단면, } r: \text{반경})
$$

- 예: r = 1.0 m, N = 500 kN, M = 150 kN·m → P_eq = 500 + 2×150/1.0 = **800 kN** (4주차 보충자료 사례)
- 보수적(안전한 쪽) 근사 — 원통 둘레 전체를 가장 불리한 한 점의 응력으로 설계

<details>
<summary>▶ 자세히 보기 — 2M/r은 어디서 오는가 (선택)</summary>

- 반경 r, 두께 t인 얇은 원통의 단면적 $A = 2\pi r t$, 단면2차모멘트 $I = \pi r^3 t$
- 굽힘에 의한 최대 응력 $\sigma_M = M r / I = M/(\pi r^2 t)$
- 축력에 의한 응력 $\sigma_N = N/(2\pi r t)$
- 최대 압축응력 $\sigma_{max} = \sigma_N + \sigma_M = \dfrac{1}{2\pi r t}\left(N + \dfrac{2M}{r}\right)$
- 즉 단면 전체에 균일하게 $N + 2M/r$의 축력이 걸린 것과 같은 최대 응력 → 이것이 등가하중 $P_{eq}$
- 전단은 같은 방식으로 $V_{eq} = V + T/(2r)$ (T: 비틀림 토크)

</details>

<div align="center"><img src="https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week04/images/loads-fig-14.png" width="720"></div>

<div align='center'><i>등가하중 개념과 단면별 공식</i></div>

## 2.4 풍선 효과 — 탱크 내압은 적이자 친구

- 탱크 내압 p는 탱크 벽을 **길이 방향으로 당기는 힘** $p\pi r^2$을 만듦 → 압축하중을 그만큼 상쇄

$$
P_{net} = P_{eq} - p\,\pi r^2
$$

- LV-B (r = 1.85 m)에서 0.25 MPa 내압이 상쇄하는 압축력 ≈ **2.7 MN** — 위의 실습에서 확인
- 내압은 동시에 벽을 팽팽하게 만들어 **좌굴(찌그러짐)에도 강하게** 함 (PART IV에서 계산)
- 대가: 내압은 벽을 **원주 방향으로 찢으려는 힘(후프응력)** 도 만듦 → 내압을 무작정 올릴 수 없음
- 아틀라스는 이 효과를 극한까지 밀어붙여 **내압 없이는 서 있을 수도 없는** 탱크를 만든 사례

> **핵심 개념**: 추진제 탱크의 내압은 원래 엔진 펌프에 추진제를 밀어 넣기 위한 것(4주차 NPSH)이지만, 구조에게는 공짜 보강재입니다. 하나의 압력이 추진·구조 두 분야의 설계에 동시에 들어가기 때문에, 탱크 압력은 여러 팀이 함께 정하는 '시스템 변수'입니다.

---
# PART III · 얼마나 튼튼하게 — '최악의 경우'를 정하는 기술

## 3.1 최악은 하나가 아니라 확률이다 — 99/90 기준

> 발사 당일의 바람을 미리 알 수는 없습니다. 그렇다고 지구 역사상 가장 센 바람에 맞춰 만들면 로켓은 너무 무거워져 아무것도 싣지 못합니다. 그래서 하중 엔지니어는 수천 번의 모의비행을 돌려 보고 "100번 날면 99번은 이 값 아래에 있다"고 말할 수 있는 선을 찾습니다. 다만 모의비행 횟수가 유한하니 그 말 자체에도 오차가 있어, "그 주장이 90% 확률로 맞다"는 단서를 붙입니다. 이것이 99/90 기준입니다.

- **포용확률 99%** (probability of enclosure): 실제 비행 하중이 이 값을 넘지 않을 확률
- **신뢰도 90%** (confidence): 유한한 표본으로 추정한 그 값이 진짜 99% 선 이상일 확률
- 대기 불확실성·궤적 편차·추력 산포를 넣은 **몬테카를로 시뮬레이션**으로 산출
- 돌풍·추력진동·버펫처럼 성격이 다른 하중은 단순 합이 아니라 **통계적 결합식**으로 묶음

<div align="center"><img src="https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week04/images/loads-fig-10.png" width="720"></div>

<div align='center'><i>99/90 통계 기준과 한계하중·극한하중</i></div>

## 3.2 [인터랙티브] 몬테카를로로 한계하중 정하기

- 비행 1회당 최대 qα를 확률분포에서 뽑아 **n회의 모의비행**을 흉내 냄
- 한계하중(limit load) = **평균 + k × 표준편차**, k는 표본 수 n에 따라 달라지는 공차계수

**해 볼 것**
- n = 10과 n = 2000에서 k는 각각 얼마인가? 같은 바람 통계인데 한계하중이 왜 달라지는가?
- 난수 시드만 바꿔 보면 n이 작을 때 결과가 얼마나 흔들리는가?

In [ ]:
w6_statistical_loads_explorer()

- **데이터가 적으면 마진이 커짐** → 같은 안전 수준을 지키려면 더 무겁게 만들어야 함
  - 고층 바람 관측 자료, 비행 계측 자료가 쌓일수록 하중이 줄고 → 구조가 가벼워지고 → 탑재량이 늚
  - 비행 데이터가 **눈에 보이지 않는 자산**이 되는 이유 — 신규 발사체가 초기에 보수적인 이유이기도 함

## 3.3 한계하중 → 극한하중 → 안전여유

| 용어 | 정의 | 의미 |
|---|---|---|
| **한계하중** (Limit) | 99/90 기준으로 정한 '예상 최대' 하중 | 영구변형 없이 견뎌야 함 |
| **극한하중** (Ultimate) | 한계하중 × 안전계수(FS) | 파괴되지 않아야 함 |
| **안전계수** FS | 무인 발사체 **1.25**, 유인 발사체 **1.4** (대표값) | 제작 편차·해석 불확실성에 대한 완충 |
| **안전여유** MS | 허용하중 / (FS × 적용하중) − 1 | **0 이상이면 통과**, 음수면 재설계 |

$$
MS = \frac{\text{허용 하중(또는 응력)}}{FS \times \text{한계 하중(또는 응력)}} - 1 \;\geq\; 0
$$

- MS가 너무 크면 **무거운 설계** (마진 낭비), 음수면 **위험한 설계** → 목표는 '0보다 조금 큰' 값
- 하중 → 사이징 → 상세해석 → 시험 → 하중 재정의의 **반복 루프**로 MS를 수렴시킴 (아래 그림)

<div align="center"><img src="https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week04/images/loads-fig-06.png" width="620"></div>

<div align='center'><i>하중-사이징-해석-시험의 반복 설계 루프</i></div>

<details>
<summary>▶ 자세히 보기 — 하중 사이클과 커플드 하중해석(CLA) (선택)</summary>

- **하중 사이클**: 예비 하중 → 예비 설계 → 구조 모델 → 최종 하중 → 검증 하중의 단계적 반복 (SMC-S-004)
  - 설계를 바꾸면 강성·질량이 바뀌고, 그러면 하중이 다시 바뀌는 **상호의존** 구조
- **커플드 하중해석(CLA)**: 발사체 유한요소모델 + 위성 모델 + 유체(슬로싱) + 자세제어(오토파일럿)를 묶은 통합 동역학 시뮬레이션
  - 결과: 비행 사건별 각 스테이션의 축력·전단력·굽힘모멘트 **시간이력**
  - 위성 제작사에 탑재체 인터페이스 하중을 넘겨주는 근거 → 발사서비스 계약의 기술 산출물
- **모달 서베이**: 실제 기체를 두드려(가진) 고유진동수와 모드형상을 재고, 해석모델을 실물에 맞춰 보정(앵커링)

</details>

<div align="center"><img src="https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week04/images/loads-fig-17.png" width="720"></div>

<div align='center'><i>커플드 하중해석(CLA) 통합 시스템</i></div>

---
# PART IV · 어떻게 버티게 만드는가 — 구조 형식과 재료

## 4.1 얇은 탱크가 실패하는 세 가지 방식

> 빈 음료캔으로 해 보는 세 가지 실험을 떠올려 보세요. 첫째, 캔 안에 압력을 계속 넣으면 결국 옆구리가 길게 찢어집니다. 둘째, 빈 캔 위에 올라서면 벽이 순식간에 주름지며 주저앉습니다. 셋째, 캔 옆면을 연필로 꾹 누르면 그 자리만 움푹 들어갑니다. 발사체 탱크도 정확히 이 세 가지 방식으로 실패합니다.

| 실패 방식 | 음료캔 실험 | 지배 변수 | 공식 (대표) |
|---|---|---|---|
| **① 터짐** (내압 파열) | 압력을 넣어 찢어짐 | 재료 강도, 두께, **용접부 강도** | 후프응력 $\sigma_h = pr/t$ |
| **② 좌굴** (전체 찌그러짐) | 위에서 밟아 주저앉음 | **강성(E)**, 두께, 형상 불완전성 | $\sigma_{cr} \approx 0.6\,\gamma\,E\,t/r$ |
| **③ 국부 실패** (국부좌굴·크리플링) | 연필로 눌러 움푹 | 국부 두께, 리브 형상, 부착물 | 격자 크기·리브 형상비 |

- 후프응력(원주 방향)은 축응력(길이 방향, $pr/2t$)의 **2배** → 터질 때는 길이 방향으로 길게 찢어짐 (소시지가 세로로 터지는 이유)
- 얇은 원통의 좌굴은 이론값보다 **훨씬 낮은 하중**에서 일어남 → 작은 찌그러짐(형상 불완전성)에 극도로 민감
  - 이를 반영하는 보정계수가 **knockdown factor γ** — 얇을수록 작아짐 (실험 기반, NASA SP-8007)

<div align="center"><img src="https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week04/images/loads-fig-21.png" width="560"></div>

<div align='center'><i>얇은 원통 압력용기의 후프응력과 축응력 유도</i></div>

## 4.2 [인터랙티브] 탱크 벽 점검 — 내압과 좌굴, 둘 다 통과해야

- LV-B 1단 탱크(반경 1.85 m)의 벽을 대상으로 **내압 강도(후프응력)** 와 **좌굴**을 동시에 점검
- 재료를 고르고 **벽두께 · 탱크 내압 · 설계 압축하중**을 바꿔 봄

**해 볼 것**
- 기본값(Al 2219, 4 mm, 0.25 MPa, 2.5 MN)에서 **빈 탱크** 좌굴 MS가 음수인 이유는?
- 탱크 내압을 0.4 MPa로 올리면 좌굴 MS와 내압 MS는 각각 어떻게 변하는가? — 둘이 **반대로 움직이는** 것을 확인
- 같은 조건에서 재료만 스테인리스 304L로 바꾸면? 무엇이 좋아지고 무엇이 나빠지는가?

In [ ]:
w6_cylinder_explorer()

- **두 가지 실패 기준이 서로 다른 방향으로 두께를 요구**
  - 비행 중(가압): 내압 강도가 두께를 정하기 쉬움
  - 지상 취급·무가압 상태: 좌굴이 두께를 정하기 쉬움 → 그래서 많은 탱크가 **운반·보관 중에도 가압** 상태를 유지
- **용접부가 약한 고리**: 같은 재료도 용접한 곳은 강도가 떨어짐(이 실습에서는 용접효율로 반영) → 용접부만 두껍게 남기는 설계, 강도 저하가 적은 용접법(PART V)이 중요

<details>
<summary>▶ 자세히 보기 — 좌굴 공식과 knockdown factor (선택)</summary>

- 축압축을 받는 얇은 원통의 고전 좌굴응력: $\sigma_{cl} = \dfrac{E}{\sqrt{3(1-\nu^2)}}\dfrac{t}{r} \approx 0.605\,E\,\dfrac{t}{r}$
- 실제 원통은 미세한 형상 불완전성 때문에 훨씬 일찍 좌굴 → NASA SP-8007의 하한 보정
$$
\gamma = 1 - 0.901\left(1 - e^{-\phi}\right), \qquad \phi = \frac{1}{16}\sqrt{\frac{r}{t}}
$$
- 예: r/t = 462(4 mm)이면 γ ≈ 0.33 → 이론값의 **1/3** 만 믿고 설계
- 내압은 무차원 압력 $\bar p = (p/E)(r/t)^2$이 클수록 좌굴계수를 높여 줌 (실습에서는 SP-8007 경향을 따르는 교육용 근사곡선 사용)
- 최근에는 정밀 형상 측정과 해석으로 더 높은 knockdown을 허용하려는 연구(NASA SBKF 프로젝트 등)가 진행 → **같은 탱크를 더 가볍게** 만드는 길

</details>

## 4.3 구조 형식 — 껍데기를 어떻게 보강할 것인가

| 형식 | 생김새 | 장점 | 단점 | 대표 사례 |
|---|---|---|---|---|
| **풍선 탱크** | 보강 없는 아주 얇은 벽 | 가장 가벼움 | 항상 가압 필요, 취급 위험 | 아틀라스, 센타우르 상단 |
| **모노코크** | 보강 없는 균일 두께 벽 | 제작 단순 | 좌굴에 약해 두꺼워짐 | 소형 상단, 압력용기 |
| **세미모노코크** (스트링거-프레임) | 외피 + 세로 보강재 + 링 | 검증된 방식, 수리 용이 | 리벳·용접 부품 많음 | 다수의 단간부·엔진부, 스트링거 용접 탱크 |
| **이소그리드 / 오소그리드** | 판을 깎아 삼각/사각 격자 | 같은 무게에 강성 수 배, 일체형 | 원소재 낭비 큼, 가공 시간 | 델타 II·IV, 아틀라스 V 공통코어부스터 |
| **샌드위치** | 두 얇은 판 + 허니콤 코어 | 굽힘강성 매우 큼 | 극저온·접합 검사 어려움 | 페어링, 단간부 |
| **복합재 일체형** | 탄소섬유 적층 | 비강성 최고, 부품 수 적음 | 극저온 미세균열·투과, 대형화 비용 | 일렉트론, 차세대 소형 발사체 |

- 탱크는 대개 **내압을 받는 부분**이라 용접 일체형(모노코크·이소그리드)이 주류, 단간부·엔진부처럼 내압이 없는 곳은 스트링거·샌드위치가 많음

<div align="center"><img src="https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week04/images/loads-fig-02.png" width="760"></div>

<div align='center'><i>발사체 주요 구조 요소와 하중 전달 경로</i></div>

## 4.4 [인터랙티브] 재료 고르기 — 같은 탱크를 다섯 가지 재료로 만든다면

- 재료 대표치 표 (상온, 교육용 근사)

<div align="center"><img src="https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week04/images/Tank_material.png" width="860"></div>

<div align='center'><i>발사체 구조재료의 기계적 특성 비교</i></div>

- 실습: LV-B 1단 LOX 탱크 원통부(지름 3.7 m, 길이 9.4 m)를 **보강 없는 모노코크**로 만들 때 재료별 필요두께와 벽 질량
- 기본값은 **무가압 보수 설계** (지상 취급 상태 가정) — 체크박스를 켜면 비행 중 가압 효과 반영

**해 볼 것**
- 무가압일 때 가장 가벼운 재료와 가장 무거운 재료는? 각각 무엇이 두께를 정하는가(지배 기준)?
- 가압 효과를 켜면 순위가 어떻게 바뀌는가?

In [ ]:
w6_material_compare_explorer()

### 이야기 — 왜 스타십은 '무거운' 스테인리스로 돌아갔나

> 스페이스X는 처음에 스타십을 탄소섬유 복합재로 만들려고 거대한 맨드릴(성형틀)까지 제작했습니다. 그런데 2019년 1월, 일론 머스크는 스테인리스강으로 바꾸겠다고 발표했습니다. 그가 든 이유는 질량표에 나오지 않는 숫자들이었습니다. 탄소섬유는 kg당 약 135달러인데 재단 과정에서 35%를 버려 실제로는 200달러 가까이 들지만, 스테인리스는 kg당 약 3달러라는 것이었습니다. 또 알루미늄이나 탄소섬유는 정상 운용 온도가 150 °C 남짓에 묶이지만 스테인리스는 800 °C 이상을 견뎌 재진입 열에도 유리하다고 설명했습니다.

- **질량만 보면 최악, 시스템으로 보면 합리적**일 수 있다는 사례
  - 재료비·스크랩률 → 기체 단가
  - 고온 강도 → 열차폐 질량 절감 (재진입하는 재사용 기체)
  - 극저온 인성·용접 용이성 → 빠른 제작과 반복 시험 (PART VI)
- 반대로 **일렉트론**처럼 작은 발사체는 복합재로 1 kg이라도 줄이는 것이 사업성의 핵심 → 규모와 사업모델이 재료를 고름
- 금속의 비강성(E/ρ)은 알루미늄·강 모두 비슷 → 차이는 **강도, 온도, 가공·용접성, 가격**에서 갈림

## 4.5 [인터랙티브] 이소그리드 — 두꺼운 판을 깎아 삼각 격자를 남기다

> 이소그리드는 1970년대 NASA가 정리한 설계법으로, 두꺼운 알루미늄 판의 안쪽을 기계로 깎아 삼각형 리브만 남기는 방식입니다. 삼각형 격자는 어느 방향으로 당겨도 성질이 같아(등방성) 계산이 쉽고, 같은 무게의 평평한 판보다 휘는 데 훨씬 강합니다. 대신 산 재료의 대부분을 칩으로 깎아 버려야 합니다.

<div align="center"><img src="https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week04/images/Tank_isogrid.jpg" width="620"></div>

<div align='center'><i>이소그리드 가공된 탱크 배럴 패널</i></div>

- 핵심 치수: 외피두께 **t**, 리브 깊이 **d**, 리브 폭 **b**, 격자 한 변 **a** (NASA CR-124075 Isogrid Design Handbook)
- 질량 등가두께 $\bar t = t(1 + 3\alpha)$, 리브 면적비 $\alpha = \dfrac{b\,d}{t\,h}$, 격자 높이 $h = \dfrac{\sqrt3}{2}a$

**해 볼 것**
- 리브 깊이 d를 키우면 강성비와 칩 비율은 각각 어떻게 되는가? — **강성 vs 원소재 낭비**의 교환관계
- 리브 폭 b를 1 mm 근처로 줄이면 어떤 경고가 나오는가?

In [ ]:
w6_isogrid_explorer()

- 이소그리드 설계는 **세 가지 실패 방식을 각각 따로 점검**함 (4주차 보충자료 사례 연구)
  - ① 격자 사이 외피의 **국부 좌굴** → 격자 크기 a를 줄이거나 외피 t를 늘림
  - ② 리브 자체의 **크리플링**(얇은 리브가 접힘) → 리브 깊이/폭 비(d/b)를 제한
  - ③ 탱크 전체의 **일반 좌굴** → 격자를 '번진 두께'로 바꿔 원통 좌굴 계산
- 가공 시간과 원소재 비용이 크다는 단점 → 최근에는 **스트링거를 용접**하거나(FSW) 처음부터 **보강재가 붙은 압출재**를 쓰는 등 제조비를 줄이는 방향으로 진화

<div align="center"><img src="https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week04/images/loads-fig-20.png" width="820"></div>

<div align='center'><i>이소그리드의 세 가지 파손모드 검증</i></div>

---
# PART V · 공장 이야기 — 탱크는 어떻게 만들어지나

## 5.1 탱크 기술 70년 — 더 얇게, 더 크게, 더 싸게

In [ ]:
w6_tank_history_timeline()

- **V-2 → 레드스톤**: 탱크를 외피 안에 따로 넣던 방식에서, **탱크 벽이 곧 기체 동체**인 통합 탱크로 → 구조질량 대폭 절감
- **아틀라스**: 통합 탱크를 극한까지 얇게 → 풍선 탱크
- **새턴 V S-II**: 두 탱크가 돔 하나를 같이 쓰는 **공통격벽** → 길이·질량 절감 (4주차 탱크 배치 참고)
- **셔틀 SLWT → 팰컨 9**: 더 가볍고 강한 **알루미늄-리튬 합금**과, 이를 강도 손실 적게 잇는 **마찰교반용접**
- **일렉트론 · 스타십**: 규모와 사업모델에 따라 **복합재**와 **스테인리스**로 갈라진 선택
- **누리호**: 탱크 제작을 국내 기업(KAI)이 맡아 **제조 공정 자체를 국산화**한 사례 (5.4)

## 5.2 공정 흐름 — 판재에서 탱크 한 통까지

In [ ]:
w6_tank_process_flow()

| 단계 | 하는 일 | 왜 어려운가 |
|---|---|---|
| ① 원소재 | 대형 알루미늄 합금 판재 구매, 성분·강도 인증 | 대형 판재는 공급처가 적고, 재료 이력 추적이 필수 |
| ② 깎기 | 기계 밀링(이소그리드) 또는 화학 밀링(부분 두께 감소) | 판 두께의 대부분을 깎아냄 → 가공 시간·칩 비용 |
| ③ 굽히기 | 롤 성형으로 원통 패널, 돔은 스피닝 또는 꽃잎 모양 패널(고어) 용접 | 깎은 판을 휘면 리브가 뒤틀림 → 깎기 전/후 순서 결정이 기술 |
| ④ 길이 방향 용접 | 패널 여러 장을 이어 원통 배럴 | 용접 열 변형으로 원통이 틀어짐 |
| ⑤ 원주 용접 | 돔·Y링·배럴을 잇는 둘레 용접 | 지름 3~10 m의 진원도 유지, 대형 치구 |
| ⑥ 비파괴검사 | X선·초음파·침투탐상 | 놓친 결함 크기가 곧 수명(PART VI) |
| ⑦ 세척 | LOX 호환 청정 — 유분·입자 제거 | 순산소 속 유기물은 발화원 |
| ⑧ 시험 | 보증압(proof)·누설·극저온 시험 | 시험 자체가 하나의 시설 투자 |

## 5.3 용접 이야기 — 녹여 붙이기 vs 비벼 붙이기

> 알루미늄 합금, 특히 가볍고 강한 알루미늄-리튬 합금은 녹여서 붙이면 용접부가 약해지고 기공이나 균열이 생기기 쉽습니다. 1991년 영국 용접연구소(TWI)에서 전혀 다른 방법이 발명되었습니다. 회전하는 공구를 두 판의 이음매에 꾹 눌러 넣고 움직이면, 마찰열로 금속이 녹지 않고 말랑해진 상태에서 반죽처럼 섞여 붙습니다. 이것이 마찰교반용접(FSW, Friction Stir Welding)입니다.

| 구분 | 용융 용접 (TIG, VPPA 등) | 마찰교반용접 (FSW) |
|---|---|---|
| 원리 | 금속을 녹여 용가재와 함께 굳힘 | 녹이지 않고 고체 상태로 휘저어 접합 |
| 용접부 강도 | 모재 대비 크게 저하되기 쉬움 | 저하가 상대적으로 적음 |
| 결함 | 기공·고온균열 위험 | 기공 거의 없음, 대신 공구 자국·접합 불량 관리 |
| 변형 | 열 입력이 커 변형 큼 | 열 입력이 작아 변형 적음 |
| 설비 | 비교적 작음 | 큰 누르는 힘 → **대형·고강성 설비**와 치구 필요 |

- 현대 대형 발사체 탱크(델타·아틀라스 계열, 팰컨 9, SLS 코어단 등)의 주력 용접법으로 자리잡음
  - NASA 미추드 조립시설의 SLS 코어단 수직 조립 용접기(VAC)는 높이 약 50 m급 FSW 설비 — **설비 자체가 국가 인프라**

<div align="center"><img src="https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week04/images/Spacex_tank.png" width="760"></div>

<div align='center'><i>추진제 탱크의 돔-원통 FSW 이음선과 원주 용접 대기 중인 돔·배럴</i></div>

<div align="center"><img src="https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week04/images/Tank_monocoque.png" width="760"></div>

<div align='center'><i>스트링거 수직 FSW, 연료탱크 배럴, 원주 FSW 설비</i></div>

## 5.4 사례 — 누리호 탱크는 어떻게 만들었나

> 누리호 1단 산화제 탱크는 지름 3.5 m, 길이 약 10 m이고 연료 탱크는 지름 3.5 m, 길이 약 6.6 m입니다. 이 탱크들은 한국항공우주산업(KAI)이 경남 사천 공장에서 2016년부터 국산화를 위해 맡아 만들었습니다. 알루미늄 합금 판을 롤러로 둥글게 말아 용접하고, 돔 모양의 덮개는 스피닝 장비로 판을 균일하고 얇게 늘려 만듭니다. 가장 어려웠던 것은 용접 열로 형태가 비틀어지는 문제였고, 이를 줄이는 공정 개발에만 3년을 들였다고 합니다. 개발 단계의 첫 내압 시험에서는 탱크가 어디까지 버티는지 확인하려고 일부러 터질 때까지 압력을 올리고, 이후 비행용 탱크에는 요구 압력보다 약 20% 높은 압력을 걸어 확인합니다. (머니투데이, 2021.11 르포 기사)

- 이 이야기에 오늘 배운 거의 모든 것이 들어 있음
  - **얇은 알루미늄 합금 통합 탱크** (PART IV) · **스피닝 돔과 롤 성형** (5.2) · **용접 변형 관리** (5.3)
  - **터뜨려 보는 인증시험** vs **요구압 이상을 거는 수락시험** (PART VI)
- 정책적 의미: 탱크 제작 국산화 = **설계도만이 아니라 공정 노하우·설비·검사 체계**를 국내에 쌓는 일 (PART VII)

---
# PART VI · 시험 이야기 — 버틴다는 것을 어떻게 증명하나

## 6.1 시험의 피라미드 — 작은 것부터 크게

| 단계 | 대상 | 목적 | 개수 |
|---|---|---|---|
| 재료 시편 | 손바닥만 한 시편, 용접 시편 | 재료·용접부의 허용응력 확정 (온도별) | 수백~수천 |
| 요소 시험 | 격자 패널, 용접 이음 | 국부좌굴·크리플링 검증, 해석모델 검증 | 수십 |
| 부분품 시험 | 돔, 배럴, Y링 | 제작 공정 검증, 압력·하중 시험 | 수 개 |
| **인증시험** (Qualification) | 비행품과 똑같이 만든 시험용 탱크 | 극한하중 이상까지 → 필요하면 **파괴까지** | 1~2 |
| **수락시험** (Acceptance) | 날아갈 모든 탱크 | **보증압(proof)** 시험·누설·비파괴검사로 제작 결함 선별 | 전수 |
| 모달 서베이 | 조립된 단/기체 | 고유진동수 측정 → 하중해석 모델 보정 | 1 |

- **보증압 시험의 숨은 목적**: 비행 압력보다 높은 압력을 한 번 걸어, 그 압력을 버텼다면 "이보다 큰 숨은 균열은 없다"는 것을 **역으로 증명** → 피로 수명 계산의 출발점(6.3)
- 보증압·파괴압 배수는 기준(NASA, ECSS, 군 표준)과 압력용기 종류마다 다름 — 보증압은 최대운용압력(MEOP)의 1.05~1.5배, 설계 파괴압은 1.25~2배 범위가 흔함

<div align="center"><img src="https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week04/images/loads-fig-18.png" width="640"></div>

<div align='center'><i>구조 검증 시험 체계 — 모달 서베이 · 인증 · 수락</i></div>

## 6.2 실패가 가르친 것들 — 세 가지 이야기

> **1963년, 서 있지 못한 로켓.** 반덴버그 기지에서 정적 시험 중이던 아틀라스-아제나는 가압 계통이 고장 나자 통째로 주저앉았습니다. 풍선 탱크의 경량화는 '압력 유지'라는 운용 조건 위에서만 성립한다는 사실을, 기체 한 대를 잃고 확인한 셈입니다.

> **2015년, 작은 볼트 하나.** 팰컨 9 CRS-7 임무는 이륙 139초 만에 2단 액체산소 탱크의 압력을 잃고 실패했습니다. 탱크 안에 고압 헬륨병을 붙잡아 두던 지지대(스트럿)의 스테인리스 아이볼트가 원인으로 지목되었는데, 10,000파운드를 견디도록 정격된 부품이 약 2,000파운드에서 부러졌습니다. NASA 독립조사팀은 항공우주 등급이 아닌 산업용 등급 부품을 충분한 선별·시험 없이 쓴 설계 오류를 가장 유력한 원인으로 보았습니다. 거대한 탱크보다 그 안의 작은 부착물이 하중 경로의 약한 고리였던 것입니다.

> **2020년, 일부러 터뜨리는 문화.** 스타십 초기 시제기 SN1은 2020년 2월 극저온 가압 시험 중 파열되었습니다. 스페이스X는 이후에도 시험용 탱크를 의도적으로 터질 때까지 가압하며 용접과 설계를 빠르게 고쳐 나갔습니다. 해석을 오래 다듬는 대신 싸게 만들어 빨리 부숴 보는 이 방식은, 4.4에서 본 값싸고 용접하기 쉬운 스테인리스 선택과 맞물려 있습니다.

- 교훈 정리
  - **운용 조건도 구조 설계의 일부** — 가압 유지, 취급 절차가 무너지면 설계 가정도 무너짐
  - **하중 경로의 모든 부품** — 탱크 벽만이 아니라 브래킷·볼트·부착물까지 등급·시험을 관리 (공급망 관리 = 구조 신뢰성)
  - **시험 철학은 사업모델의 선택** — 해석 중심(적은 시험, 긴 일정) vs 반복 시험 중심(많은 시제기, 빠른 학습)

## 6.3 [인터랙티브] 재사용이 구조에 주는 새 숙제 — 피로

- 1회용 탱크는 '한 번만 버티면' 되지만, 재사용 탱크는 **충전·가압·비행·착륙을 수십 번 반복** → 작은 균열이 조금씩 자람
- 균열 성장은 **Paris 법칙**으로 근사

$$
\frac{da}{dN} = C\,(\Delta K)^m, \qquad \Delta K = Y\,\Delta\sigma\,\sqrt{\pi a}
$$

- 균열이 **임계 크기**(파괴인성 K_IC로 정해짐)에 이르면 순간 파단 → 그 전까지의 사이클 수가 수명
- 출발점인 **초기 균열 크기** = 비파괴검사로 '없다고 보장할 수 있는' 최소 결함 크기

**해 볼 것**
- 반복 응력을 250 → 200 MPa로 20%만 낮추면 허용 비행 횟수는 몇 배가 되는가?
- 검사 정밀도를 높여 초기결함을 1.5 → 0.5 mm로 보장하면?

In [ ]:
w6_reuse_fatigue_explorer()

- **응력 20% 감소 → 수명 2배 이상** (지수 3.2의 효과): 재사용 기체는 1회용보다 약간 두껍게, 응력을 낮게 설계하는 경향
- **검사 기술이 재사용 횟수를 정함** → 비행 사이 점검(turnaround) 시간과 비용이 곧 재사용 경제성 (5주차·12주차 연결)
- 실습의 재료 상수는 알루미늄 합금급 **교육용 대표치**이며, 실제로는 용접부·극저온·하중 순서 효과를 시험으로 확인

---
# PART VII · 정책과 사업의 눈으로 — 구조는 왜 엔지니어만의 문제가 아닌가

## 7.1 구조비는 곧 사업성이다

- 4주차 LV-B 계산 다시 보기: 구조질량비 ε를 0.01 올렸을 때 GLOW 변화
  - 1단 ε +0.01 → GLOW **+3.4%** / 2단 ε +0.01 → GLOW **+14.6%**
- 같은 탑재체를 위해 더 큰 발사체 → 엔진·추진제·제작비·발사장 비용이 모두 증가
- 따라서 **하중을 정확히 아는 것(데이터)**, **좌굴 knockdown을 덜 보수적으로 쓸 수 있는 검증 능력**, **가벼운 재료와 용접 기술**이 모두 kg/달러로 환산되는 경쟁력

## 7.2 제조 인프라는 산업정책이다

| 요소 | 왜 진입장벽인가 | 정책적 함의 |
|---|---|---|
| 대형 성형·용접 설비 | 지름별 전용 치구, 수십 m급 FSW 설비 | 한 번 정한 **지름이 수십 년 인프라를 결정** (4주차 지름 선택) |
| 재료 공급망 | 대형 Al-Li 판재 등 공급처 제한, 재료 인증 | 수출통제·단일 공급처 리스크 → 국산화·다변화 |
| 공정 노하우 | 용접 변형 관리 같은 암묵지 (누리호 3년) | 사업 간 **연속 물량**이 없으면 인력·노하우 소실 |
| 시험 시설 | 극저온·고압 시험장, 구조 시험장 | 공공 시설 개방 vs 기업 자체 구축의 역할 분담 |

- 누리호의 탱크 제작이 연구기관에서 기업(KAI)으로 넘어간 것은 **기술 이전 + 공급망 형성**의 사례 → 후속 물량(반복 발사, 차세대 발사체)이 이어져야 노하우가 유지됨

## 7.3 시험 철학은 위험을 누가 지는가의 문제

- 해석·문서 중심 인증: 시제기가 적어 초기 비용 절감, 대신 일정이 길고 실패 1회의 충격이 큼
- 반복 시험 중심 개발: 시제기를 많이 부수며 빨리 학습, 대신 **실패를 감당할 자본과 규제 환경**이 필요
- 규제기관(발사허가, 7주차)은 **공공 안전**을 보고, 고객·보험사(12주차)는 **임무 성공**을 봄 → 같은 구조 데이터가 서로 다른 질문에 쓰임

## 7.4 사업기획보고서 연결 — 기술타당성 절에 넣을 것

- [ ] 우리 발사체의 **지름**과 그 근거 (수송·제작 설비·전장, 4주차) — 국내 설비로 만들 수 있는가?
- [ ] **탱크 재료·구조 형식·접합 공법**의 선택과 이유 (질량 vs 단가 vs 제작 속도)
- [ ] 구조질량비 ε 가정치와 **민감도** — 상단 구조가 1% 무거워지면 사업성은?
- [ ] 개발 시험 계획: **인증 시험체 수**, 시험 시설 확보 방안(자체 vs 공공)
- [ ] 재사용이라면: 목표 재사용 횟수를 뒷받침할 **피로·검사 전략**과 비행 간 점검 비용

## 토론 질문

1. 스타십의 스테인리스 선택과 일렉트론의 복합재 선택은 모두 합리적일 수 있다. **두 기업의 사업모델 차이**로 이 선택을 설명해 보라.
2. 한국이 차세대 발사체의 지름을 정할 때, **기존 누리호 설비를 재활용**하는 것과 **새 지름으로 투자**하는 것의 장단점은?
3. 비행 데이터가 쌓일수록 구조 마진을 줄일 수 있다면, 신규 발사체 기업이 불리한 출발선을 극복하도록 **공공이 제공할 수 있는 데이터·시설**은 무엇인가?

---
## 오늘 남길 여섯 문장

1. **발사체 탱크는 음료캔보다 상대적으로 얇다** — 내압, 보강 격자, 재료가 그 껍데기를 버티게 한다.
2. **하중에는 하나의 최악이 없다** — 굽힘은 Max-Q 근처에서, 축 압축은 연소종료 직전에 최대가 되며, 비행 중 중력은 내력을 만들지 않는다.
3. **발사체를 막대 하나로 보면 하중이 숫자가 된다** — 단면을 잘라 축력·전단력·굽힘모멘트를 구하고, 이를 등가 압축하중 N + 2M/r 로 묶는다.
4. **설계하중은 확률로 정한다** — 99/90 한계하중에 안전계수를 곱하며, 데이터가 적을수록 마진은 커진다.
5. **얇은 탱크는 터지기 전에 찌그러진다** — 내압은 좌굴을 막아 주지만 후프응력을 키우므로, 두께는 두 실패 기준의 줄다리기로 정해진다.
6. **탱크는 설계만큼 제작과 시험의 산물이다** — 용접 기술, 검사 정밀도, 시험 철학이 질량·수명·일정·비용을 결정하고, 그 능력은 곧 산업 인프라다.

## 자가진단 퀴즈

<details>
<summary>Q1. 비행 중 발사체 내부의 축력 계산에 중력이 들어가지 않는 이유는?</summary>

비행 중에는 기체 전체가 함께 중력 가속도를 받으므로 중력은 각 부분 사이에 힘을 주고받게 만들지 않는다(자유낙하하는 엘리베이터 안에서 무게를 느끼지 않는 것과 같다). 위쪽 질량을 아래로 누르는 내력은 추력에서 항력을 뺀 힘이 만든 가속도 $a_x = (T-D)/m$에서만 생기므로, 추진제가 소진되어 가속이 최대가 되는 연소종료 직전에 축 하중이 가장 크다. 반면 발사대 위에 서 있을 때는 지면이 중력을 받쳐 주므로 중력이 내력을 만든다.
</details>

<details>
<summary>Q2. 반경 1.85 m, 축력 2.0 MN, 굽힘모멘트 0.5 MN·m인 단면의 등가 압축하중은? 탱크 내압 0.2 MPa를 반영하면?</summary>

$P_{eq} = N + 2M/r = 2.0 + 2(0.5)/1.85 = 2.54$ MN. 내압이 상쇄하는 힘은 $p\pi r^2 = 0.2\times10^6 \times \pi \times 1.85^2 \approx 2.15$ MN이므로 순 압축하중은 약 0.39 MN으로 줄어든다. 내압이 구조에 큰 도움이 된다는 것을 보여 주며, 반대로 내압이 빠진 상태(지상 취급 등)가 더 위험할 수 있음을 뜻한다.
</details>

<details>
<summary>Q3. 같은 바람 통계에서 모의비행을 30회 돌린 팀과 3,000회 돌린 팀의 한계하중이 다른 이유는?</summary>

99/90 한계하중은 평균 + k × 표준편차로 정해지는데, 공차계수 k는 표본 수가 적을수록 커진다. 표본이 적으면 진짜 99% 선을 추정하는 데 오차가 크기 때문에, 90% 신뢰도를 지키려면 더 큰 값을 써야 한다. 그래서 30회 팀은 더 보수적인(큰) 한계하중을 얻고 더 무거운 구조를 만들게 된다.
</details>

<details>
<summary>Q4. 탱크 내압을 올리면 좋아지는 것과 나빠지는 것을 각각 설명하라.</summary>

좋아지는 것: 길이 방향 인장력 $p\pi r^2$이 압축하중을 상쇄하고, 벽이 팽팽해져 좌굴 강도가 높아진다. 또 엔진 펌프 입구의 NPSH가 커진다(4주차). 나빠지는 것: 후프응력 $pr/t$가 커져 터짐에 대한 두께 요구가 늘고, 가압가스 질량과 가압계통 부담이 커지며, 용접부처럼 약한 곳의 여유가 줄어든다.
</details>

<details>
<summary>Q5. 마찰교반용접(FSW)이 알루미늄-리튬 합금 탱크에 널리 쓰이는 이유는?</summary>

FSW는 금속을 녹이지 않고 고체 상태에서 휘저어 접합하므로, 녹였다 굳히는 용융 용접에서 생기기 쉬운 기공·고온균열이 거의 없고 용접부 강도 저하와 열 변형도 상대적으로 작다. 용융 용접이 까다로운 Al-Li 합금의 장점(가볍고 강함)을 살릴 수 있어 대형 탱크의 주력 용접법이 되었다. 대신 큰 누르는 힘을 견디는 대형·고강성 설비와 치구가 필요하다.
</details>

<details>
<summary>Q6. 재사용 탱크에서 비파괴검사의 정밀도가 사업성과 연결되는 이유는?</summary>

피로 수명은 '검사로 없다고 보장할 수 있는 최소 결함 크기'에서 출발해 균열이 임계 크기까지 자라는 사이클 수로 정해진다. 검사가 정밀해 초기 결함을 작게 보장할수록 허용 비행 횟수가 늘어나고, 검사 주기를 늘려 비행 간 점검 시간과 비용을 줄일 수 있다. 재사용 횟수와 점검 비용은 곧 발사 1회당 비용이므로, 검사 기술은 재사용 경제성의 핵심 변수다.
</details>

## 참고문헌 및 더 읽을거리

★ 표시는 이번 주 필수 확인 자료

**하중해석**
- ★ 4주차 보충자료 `week04/week04-loads-analysis.ipynb` — 발사체 구조 하중 해석과 예비 구조 설계
- SMC-S-004 (2008), *Independent Structural Loads Analysis*, USAF Space and Missile Systems Center
- NASA-STD-5002A, *Load Analysis of Spacecraft and Payloads*
- ECSS-E-HB-32-26, *Spacecraft Mechanical Loads Analysis Handbook*
- NASA SP-8009, *Propellant Slosh Loads*

**구조 설계·좌굴**
- ★ NASA SP-8007, *Buckling of Thin-Walled Circular Cylinders* — knockdown factor의 출처
- NASA CR-124075, *Isogrid Design Handbook*
- NASA-STD-5001, *Structural Design and Test Factors of Safety for Spaceflight Hardware*
- Henson, G., *Materials for Launch Vehicle Structures*, NASA NTRS 20170001809

**압력용기·피로**
- Huzel, D. K. & Huang, D. H., *Design of Liquid Propellant Rocket Engines*, NASA SP-125 — 탱크 설계 장
- NASA-STD-5019, *Fracture Control Requirements for Spaceflight Hardware*

**이야기 속 사례**
- [Balloon tank — Wikipedia](https://en.wikipedia.org/wiki/Balloon_tank) (아틀라스 탱크 두께·1963년 붕괴)
- [Space.com (2019.1.23), Why Elon Musk Turned to Stainless Steel for SpaceX's Starship](https://space.com/43101-elon-musk-explains-stainless-steel-starship.html)
- [SpaceX CRS-7 — Wikipedia](https://en.wikipedia.org/wiki/SpaceX_CRS-7) · [NASASpaceflight (2018), NASA's IRT publishes report on SpaceX's CRS-7 failure](https://www.nasaspaceflight.com/2018/03/nasas-irt-publishes-crs-7-failure/)
- [머니투데이 (2021.11), 누리호 탱크 제작 현장 르포](https://www.mt.co.kr/industry/2021/11/17/2021111617343194944)

> 실습의 모든 수치(재료 상수, 용접효율, 좌굴 내압 보정, 피로 상수, 질량 분포)는 개념 이해를 위한 **교육용 근사치**입니다. 실제 설계에는 인증된 재료 허용치와 해당 사업의 설계 기준을 사용해야 합니다.